# Inspect the leaderboard and official AutoML notebooks

[Guide](README.md) · [00: prepare data](00_prepare_data.ipynb) · [01: experiment](01_experiment.ipynb) · [02: results](02_results.ipynb) · [03: predictor](03_predictor.ipynb)

Complete monitoring of a successful run in [01 — Experiment](01_experiment.ipynb). This notebook reads pipeline artifacts, checks that the uploaded external test CSV was used in full, and selects the highest score for the run's recorded metric. It uses the saved run's data and settings even if `.env` has changed, and never retrains a model.

In [1]:
from pathlib import Path
import sys

# Works when Jupyter starts in either the repo root or automl/.
automl_dir = Path.cwd() if Path('workflow.py').exists() else Path.cwd() / 'automl'
sys.path.insert(0, str(automl_dir.resolve()))
import pandas as pd
from IPython.display import display, Markdown, FileLink
import workflow as wf

settings = wf.settings()  # Never display this mapping: it may contain credentials.

In [ ]:
run = wf.read('run.json')
print('Inspecting run:', run['run_id'], '| State:', run.get('state', 'unknown'))
print('Ranking metric from that run:', run['arguments']['eval_metric'])
print('External test data:', run['data']['datasets']['test']['uri'])
print('Expected external test rows:', run['data']['datasets']['test']['rows'])
print('Artifact location:', f"s3://{run['artifact_bucket']}/{run['artifact_prefix']}")

In [ ]:
s3 = wf.s3_client()
results = wf.collect_results(s3)
columns = ['model', 'roc_auc', 'f1', 'precision', 'recall', 'accuracy']
leaderboard = pd.DataFrame(results['models'])
display(leaderboard[[c for c in columns if c in leaderboard.columns]])
print('Ranking metric:', results['metric'])
print('External test score:', results['holdout_score'])
print('Verified external test rows:', results['holdout_rows'])
print('Selected predictor:', results['predictor_uri'])

The external test set participates in candidate evaluation/ranking in the managed pipeline. This score is not an untouched final generalization estimate. Accuracy is shown only as context; use ROC-AUC or F1 to discuss this imbalanced problem.

In [6]:
leaderboard_path = wf.OUTPUTS / 'leaderboard.html'
if results['leaderboard_html'] and leaderboard_path.exists():
    # Open the original generated HTML in a separate tab.
    display(FileLink(str(leaderboard_path.relative_to(Path.cwd()))))
else:
    print('HTML leaderboard not found; the metrics table above is available.')

/Users/mszewczy/projects/fraud-detection/automl/outputs/leaderboard.html

## Open the official generated notebooks

The managed pipeline emits a run-level **experiment notebook** and a per-model **predictor notebook**. These links open copies from this run's S3 artifacts. The experiment notebook can start another run; inspect its parameters before running it. The predictor notebook loads the selected serialized predictor and can run local `predict_proba`.

Use the same Connection credentials and set `AWS_S3_BUCKET` to the artifact bucket. Follow the generated notebooks' version-specific dependency cells in a separate kernel/workbench if needed. Notebook 03 uses the serving runtime for REST predictions without installing AutoGluon locally.

In [7]:
for label, relative_path in results['notebooks'].items():
    print(label)
    display(FileLink(str((wf.ROOT / relative_path).relative_to(Path.cwd()))))

automl_experiment_notebook.ipynb


/Users/mszewczy/projects/fraud-detection/automl/outputs/generated/automl_experiment_notebook.ipynb

automl_predictor_notebook.ipynb


/Users/mszewczy/projects/fraud-detection/automl/outputs/generated/automl_predictor_notebook.ipynb

In [ ]:
print(wf.ticket_report())

Next: [03_predictor.ipynb](03_predictor.ipynb)